In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import kagglehub

c:\Users\piotr\venv39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\piotr\.cache\kagglehub\datasets\kmader\skin-cancer-mnist-ham10000\versions\2


In [3]:
def compute_sharpness(gray):
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def compute_contrast(gray):
    return gray.std()

def compute_brightness(gray):
    return gray.mean()


In [4]:
from glob import glob

def analyze_images(metadata_csv, image_root):
    df = pd.read_csv(metadata_csv)

    image_paths = {
        os.path.splitext(os.path.basename(p))[0]: p
        for p in glob(os.path.join(image_root, "**", "*.jpg"), recursive=True)
    }

    records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_path = image_paths.get(row["image_id"])
        if img_path is None:
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        records.append({
            "image_id": row["image_id"],
            "label": row["dx"],
            "sharpness": compute_sharpness(gray),
            "contrast": compute_contrast(gray),
            "brightness": compute_brightness(gray),
        })

    return pd.DataFrame(records)


In [5]:
def filter_high_quality(
    df,
    sharpness_q=0.5,
    contrast_q=0.5,
    brightness_low_q=0.2,
    brightness_high_q=0.8
):
    selected = []

    for cls in df["label"].unique():
        cls_df = df[df["label"] == cls]

        sharp_thr = cls_df["sharpness"].quantile(sharpness_q)
        cont_thr = cls_df["contrast"].quantile(contrast_q)
        bright_low = cls_df["brightness"].quantile(brightness_low_q)
        bright_high = cls_df["brightness"].quantile(brightness_high_q)

        cls_selected = cls_df[
            (cls_df["sharpness"] >= sharp_thr) &
            (cls_df["contrast"] >= cont_thr) &
            (cls_df["brightness"] >= bright_low) &
            (cls_df["brightness"] <= bright_high)
        ]

        selected.append(cls_selected)

    return pd.concat(selected).reset_index(drop=True)


In [6]:
def balanced_sampling(
    df,
    samples_per_class,
    random_state=42
):
    subsets = []

    for cls in df["label"].unique():
        cls_df = df[df["label"] == cls]

        if len(cls_df) < samples_per_class:
            print(f"Warning: class {cls} has only {len(cls_df)} samples")
            subsets.append(cls_df)
        else:
            subsets.append(
                cls_df.sample(
                    samples_per_class,
                    random_state=random_state
                )
            )

    return pd.concat(subsets).reset_index(drop=True)


In [7]:
metadata_csv = os.path.join(path, "HAM10000_metadata.csv")

quality_df = analyze_images(metadata_csv, path)



100%|██████████| 10015/10015 [01:55<00:00, 86.56it/s]


In [8]:
print(quality_df.columns)

Index(['image_id', 'label', 'sharpness', 'contrast', 'brightness'], dtype='object')


In [9]:
filtered_df = filter_high_quality(quality_df)

subset_df = balanced_sampling(
    filtered_df,
    samples_per_class=200
)

subset_df.to_csv("ham10000_high_quality_subset.csv", index=False)


In [10]:
metadata_csv = os.path.join(path, "HAM10000_metadata.csv")

meta_df = pd.read_csv(metadata_csv)

df = quality_df.merge(
    meta_df[["image_id", "dx", "localization"]],
    on="image_id",
    how="inner"
)

print("Merged rows:", len(df))
print(df.head())


Merged rows: 10015
       image_id label   sharpness   contrast  brightness   dx localization
0  ISIC_0027419   bkl   62.193005  18.175079  178.635804  bkl        scalp
1  ISIC_0025030   bkl   68.733841  34.779243  171.041104  bkl        scalp
2  ISIC_0026769   bkl   50.287609  16.603222  175.677433  bkl        scalp
3  ISIC_0025661   bkl  220.249536  33.436485  160.159826  bkl        scalp
4  ISIC_0031633   bkl   53.105486  40.004478  183.030815  bkl          ear


In [11]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min() + 1e-8)

df["quality_score"] = (
    normalize(df["sharpness"]) +
    normalize(df["contrast"]) -
    normalize(abs(df["brightness"] - df["brightness"].median()))
)


In [12]:
SAMPLES_PER_GROUP = 5

selected_rows = []

for loc in df["localization"].dropna().unique():
    loc_df = df[df["localization"] == loc]

    for cls in loc_df["dx"].unique():
        cls_df = loc_df[loc_df["dx"] == cls]

        if len(cls_df) == 0:
            continue

        cls_df = cls_df.sort_values(
            by="quality_score",
            ascending=False
        )

        selected_rows.append(cls_df.head(SAMPLES_PER_GROUP))


In [13]:
subset_df = pd.concat(selected_rows).reset_index(drop=True)

print("Final subset size:", len(subset_df))
print(
    subset_df
    .groupby(["localization", "dx"])
    .size()
    .head(20)
)


Final subset size: 368
localization  dx   
abdomen       akiec    5
              bcc      5
              bkl      5
              df       4
              mel      5
              nv       5
              vasc     5
acral         nv       5
back          akiec    5
              bcc      5
              bkl      5
              df       2
              mel      5
              nv       5
              vasc     5
chest         akiec    5
              bcc      5
              bkl      5
              mel      5
              nv       5
dtype: int64


In [14]:
subset_df.to_csv(
    "ham10000_subset_5_per_class_per_localization.csv",
    index=False
)


In [15]:
print("Localizations:", subset_df["localization"].nunique())
print("Classes:", subset_df["dx"].nunique())


Localizations: 15
Classes: 7


In [16]:
subset_df.groupby("localization")["dx"].nunique().sort_values()


localization
acral              1
genital            3
ear                4
unknown            4
foot               5
chest              6
hand               6
face               6
trunk              6
neck               6
scalp              6
back               7
abdomen            7
lower extremity    7
upper extremity    7
Name: dx, dtype: int64

In [17]:
SAMPLES_PER_GROUP = 10

selected_rows = []

for loc in df["localization"].dropna().unique():
    loc_df = df[df["localization"] == loc]

    for cls in loc_df["dx"].unique():
        cls_df = loc_df[loc_df["dx"] == cls]

        if len(cls_df) == 0:
            continue

        cls_df = cls_df.sort_values(
            by="quality_score",
            ascending=False
        )

        q75 = cls_df["quality_score"].quantile(0.75)
        top_q_df = cls_df[cls_df["quality_score"] >= q75]

        chosen = top_q_df.head(SAMPLES_PER_GROUP)

        if len(chosen) < SAMPLES_PER_GROUP:
            remaining_needed = SAMPLES_PER_GROUP - len(chosen)

            remaining_df = cls_df[
                ~cls_df["image_id"].isin(chosen["image_id"])
            ]

            filler = remaining_df.head(remaining_needed)
            chosen = pd.concat([chosen, filler])

        selected_rows.append(chosen)


In [18]:
subset_df = (
    pd.concat(selected_rows)
    .drop_duplicates(subset="image_id")
    .reset_index(drop=True)
)

print("Final subset size:", len(subset_df))
print(
    subset_df
    .groupby(["localization", "dx"])
    .size()
    .describe()
)


Final subset size: 680
count    81.000000
mean      8.395062
std       3.011142
min       1.000000
25%      10.000000
50%      10.000000
75%      10.000000
max      10.000000
dtype: float64


In [19]:
subset_df.to_csv(
    "ham10000_subset_quality_aware_10_per_loc_dx.csv",
    index=False
)


In [20]:
def get_lesion_mask(img):
    """
    img: BGR image (OpenCV)
    returns: binary mask of lesion region
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    h, s, v = cv2.split(hsv)

    # Lesions tend to be darker and more saturated
    s_mask = s > 30
    v_mask = v < 220

    mask = np.logical_and(s_mask, v_mask).astype(np.uint8) * 255

    # Morphological cleanup
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask


In [21]:
def largest_component(mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    if num_labels <= 1:
        return None

    # Ignore background (label 0)
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])

    component = np.zeros_like(mask)
    component[labels == largest] = 255

    return component


In [22]:
def crop_lesion(img, mask, padding=0.1):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    h, w = img.shape[:2]

    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()

    # Padding in pixels
    pad_x = int((x_max - x_min) * padding)
    pad_y = int((y_max - y_min) * padding)

    x_min = max(0, x_min - pad_x)
    x_max = min(w, x_max + pad_x)
    y_min = max(0, y_min - pad_y)
    y_max = min(h, y_max + pad_y)

    return img[y_min:y_max, x_min:x_max]


In [23]:
def preprocess_image(
    img_path,
    output_path,
    target_size=(224, 224),
    padding=0.1
):
    img = cv2.imread(img_path)
    if img is None:
        return False

    mask = get_lesion_mask(img)
    mask = largest_component(mask)

    if mask is None:
        return False

    cropped = crop_lesion(img, mask, padding)

    if cropped is None or cropped.size == 0:
        return False

    cropped = cv2.resize(cropped, target_size, interpolation=cv2.INTER_AREA)
    cv2.imwrite(output_path, cropped)

    return True


In [24]:
from glob import glob

IMAGE_ROOTS = [
    os.path.join(path, "HAM10000_images_part_1"),
    os.path.join(path, "HAM10000_images_part_2"),
]

image_path_map = {}

for root in IMAGE_ROOTS:
    for img_path in glob(os.path.join(root, "*.jpg")):
        image_id = os.path.splitext(os.path.basename(img_path))[0]
        image_path_map[image_id] = img_path

print("Images indexed:", len(image_path_map))


Images indexed: 10015


In [25]:
OUTPUT_DIR = "data/selected_cropped"
os.makedirs(OUTPUT_DIR, exist_ok=True)

success, fail = 0, 0

for image_id in tqdm(subset_df["image_id"]):
    input_path = image_path_map.get(image_id)

    if input_path is None:
        fail += 1
        continue

    output_path = os.path.join(OUTPUT_DIR, image_id + ".jpg")

    ok = preprocess_image(input_path, output_path)

    if ok:
        success += 1
    else:
        fail += 1

print(f"Processed: {success}, Failed: {fail}")


100%|██████████| 680/680 [00:12<00:00, 54.30it/s]

Processed: 680, Failed: 0


In [26]:
missing = [
    img_id for img_id in subset_df["image_id"]
    if img_id not in image_path_map
]

print("Missing images:", len(missing))
print(missing[:5])


Missing images: 0
[]
